#  LangChain의 RAG 콤포넌트 - 문서 임베딩(Embeddings) 

### **학습 목표:**  임베딩 모델과 벡터 데이터베이스를 효과적으로 연동할 수 있다

### **실습 자료**: 

- data/transformer.pdf

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob  

from pprint import pprint  
import json

`(3) 문서 로드`

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

PDF 문서 개수: 15


`(4) 텍스트 분할`

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,             # 청크 크기  
    chunk_overlap=200,           # 청크 중 중복되는 부분 크기
    length_function=len,         # 글자 수를 기준으로 분할
    separators=["\n\n", "\n", " ", ""],  # 구분자 - 재귀적으로 순차적으로 적용 
)

# PDF 문서를 텍스트로 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 52
각 청크의 길이: [987, 910, 975, 452, 933, 995, 902, 907, 997, 386, 924, 955, 216, 926, 903, 950, 1000, 913, 909, 873, 945, 973, 946, 997, 196, 981, 980, 946, 939, 999, 944, 921, 734, 958, 946, 946, 617, 983, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


# 문서 임베딩(Document Embedding)

- 개념: 
    - 텍스트를 벡터(숫자 배열)로 변환하는 과정
    - 문서의 의미적 특성을 수치화하여 컴퓨터가 이해하고 처리할 수 있는 형태로 변환 

- 목적:
    - 텍스트 간 유사도 계산 가능
    - 벡터 데이터베이스 저장 및 검색
    - 의미 기반 문서 검색 구현

- LangChain의 임베딩 모델 종류:
    - OpenAI 임베딩
    - HuggingFace 임베딩 
    - Ollama 임베딩

### 1. **OpenAI**

- LangChain에서 가장 널리 사용되는 임베딩 모델 중 하나

- 주요 특징:
    1. 고품질의 임베딩 생성
    2. 다양한 언어 지원 (다국어 지원)
    3. 일관된 성능
    4. 손쉬운 통합

- 사용시 주의사항:
    1. API 키 설정이 필요 (환경 변수 OPENAI_API_KEY)
    2. API 사용량에 따른 비용 발생
    3. 긴 텍스트는 자동으로 분할되지 않으므로 필요시 TextSplitter를 사용


- 모델별 특징

    | 모델명 | 가격 효율 (페이지/1달러) | 성능 (MTEB) | 최대 입력 | 기본 차원 | 특징 및 추천 |
    | :--- | :---: | :---: | :---: | :---: | :--- |
    | **`text-embedding-3-small`** | **62,500** | 62.3% | 8,191 | 1,536 | **[가성비]** ada-002 대비 5배 저렴하고 성능 우수. 일반적인 RAG 구축 시 1순위. |
    | **`text-embedding-3-large`** | 9,615 | **64.6%** | 8,191 | 3,072 | **[고성능]** 미세한 의미 차이 구분이 중요할 때 사용. 차원 축소 기능 지원. |
    | **`text-embedding-ada-002`** | 12,500 | 61.0% | 8,191 | 1,536 | **[레거시]** 기존에 널리 쓰이던 모델이나, 현재는 v3-small 사용을 권장함. |

> **💡 Tip:** `text-embedding-3` 계열은 **Matryoshka Embedding** 기술이 적용되어, 저장 공간 절약을 위해 임베딩 차원(예: 1536 → 512)을 줄여서 요청해도 성능 하락이 매우 적습니다.

`(1) embedding 모델`

In [13]:
from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings 모델 생성
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 사용할 모델 이름
    dimensions=None, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 객체 출력
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x1294ba150>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12b397550>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [14]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model.embedding_ctx_length

8191

In [32]:
# 임베딩 모델의 임베딩 차원 확인 - 기본값 (None)
embeddings_model.dimensions

512

In [16]:
# OpenAIEmbeddings 모델 생성할 때 임베딩 차원을 지정하는 예시
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 모델의 임베딩 차원 확인 
embeddings_model.dimensions

512

In [17]:
# OpenAIEmbeddings 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 객체 출력
embeddings_openai

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x12bac2890>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12bac3590>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

`(2) embed_documents 사용`

In [18]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_openai = embeddings_openai.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_openai)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_openai[0])}")
print(document_embeddings_openai[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1536
[-0.0024314899928867817, 0.012240828014910221, -0.002483319491147995, 0.014899917878210545, 0.018433352932333946, -0.04564620926976204, -0.0032021752558648586, 0.047016315162181854, -0.018226033076643944, -0.03180091455578804, 0.0068550435826182365, 0.007021800149232149, -0.018550531938672066, -0.02711370401084423, 0.004935090895742178, -0.015819331631064415, -0.04149983078241348, 0.010041444562375546, 0.05155930295586586, -0.045898597687482834, -0.014575418084859848, -0.02767256274819374, -0.01880292035639286, -0.022570716217160225, 0.004549747798591852, -0.03991338983178139, 0.0503334179520607, 0.017730271443724632, 0.00022745922615285963, -0.02349013090133667, 0.04889119789004326, -0.01534159667789936, -0.03001616708934307, -0.06165483221411705, 0.020353306084871292, 0.03542448580265045, 0.0025666977744549513, -0.006837015971541405, -0.005128888878971338, 0.024860236793756485, 0.0074905212968587875, 0.027474258095026016, -0.0025148680433630943, 0.02610

`(3) embed_query 사용`

In [19]:
embedded_query_openai = embeddings_openai.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query_openai)}")
print(embedded_query_openai)

쿼리 임베딩 벡터의 차원: 1536
[-0.02253107912838459, 0.022246355190873146, 0.0005270341061986983, 0.005661242175847292, 0.01219563465565443, -0.044834379106760025, -0.02630840428173542, 0.0357232429087162, -0.0026716506108641624, 0.0147866141051054, -0.002117627300322056, 0.0009075545240193605, -0.004873508587479591, -0.0736483484506607, 0.0090589364990592, -0.015346569009125233, -0.059753865003585815, -0.022436171770095825, 0.02239820919930935, -0.0692446306347847, -0.029212579131126404, 0.023195432499051094, -0.036843154579401016, 0.00413085613399744, 0.011123177595436573, -0.052654772996902466, 0.010762528516352177, 3.670257228804985e-06, -0.010791000910103321, -0.03410981222987175, 0.02528340183198452, -0.018896115943789482, -0.001538690528832376, -0.05895664170384407, 0.049883466213941574, -0.003516328986734152, -0.002804521471261978, -0.008399328216910362, -0.004617257975041866, 0.02279682084918022, -0.011939384043216705, 0.037982046604156494, 0.0033241407945752144, 0.035229723900556564, -

`(4) 유사도 기반 검색`

In [20]:
from langchain_community.utils.math import cosine_similarity
import numpy as np

# 쿼리와 가장 유사한 문서 찾기 함수
def find_most_similar(
        query: str, 
        doc_embeddings: np.ndarray,
        embeddings_model  # 기본값 제거, 명시적 전달 강제
        ) -> tuple[str, float]:
    """
    쿼리와 가장 유사한 문서를 찾는 함수
    
    Args:
        query: 검색 쿼리 문자열
        doc_embeddings: 문서 임베딩 배열
        embeddings_model: 임베딩 모델 객체
    
    Returns:
        tuple: (가장 유사한 문서, 유사도 점수)
    """
    # 쿼리 임베딩: OpenAI 임베딩 사용 
    query_embedding = embeddings_model.embed_query(query)

    # 코사인 유사도 계산
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    # 가장 유사한 문서 인덱스 찾기
    most_similar_idx = np.argmax(similarities)

    # 가장 유사한 문서와 유사도 반환: 문서, 유사도
    return documents[most_similar_idx], similarities[most_similar_idx]

# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(
        query, 
        document_embeddings_openai, 
        embeddings_model=embeddings_openai
        )
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()
    

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7119

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.6817

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.7052



### 2. **Huggingface**

- LangChain에서 오픈소스 기반의 대표적인 임베딩 모델

- 주요 특징:
    1. 로컬 환경에서 실행 가능
    2. 다양한 사전학습 모델 지원
    3. 커스텀 모델 학습 및 적용 가능
    4. 무료 사용 가능 (API 비용 없음)

- 사용시 주의사항:
    1. 로컬 컴퓨팅 자원 필요 (CPU/GPU)
    2. 초기 모델 다운로드 시간 소요
    3. 메모리 사용량 고려 필요
    4. transformers 라이브러리 설치 필요

- 임베딩 벡터 특성:
    1. 모델별로 다양한 차원 제공 (128 ~ 1024)
    2. sentence-transformers 기반 구현
    3. BERT 계열 모델 구조 사용
    4. 코사인 유사도 기반 검색 최적화

- 대표적인 임베딩 모델:

    | 모델명 (Hugging Face ID) | 차원 | 언어 | 특징 및 추천 용도 |
    | :--- | :---: | :---: | :--- |
    | **`all-MiniLM-L6-v2`** | 384 | **영어** | **[영어 표준/경량]** 매우 빠르고 메모리 효율이 좋음. 영어 전용 검색/분류 작업의 입문용 모델. |
    | **`all-mpnet-base-v2`** | 768 | **영어** | **[영어 고성능]** MiniLM보다 느리지만, 문장의 뉘앙스를 가장 정확하게 포착함. 영어권 RAG의 표준. |
    | **`paraphrase-multilingual-MiniLM-L12-v2`** | 384 | 다국어 | **[다국어 경량]** 한국어를 포함한 50개국어 지원. 속도가 빨라 실시간 서비스에 적합. |
    | **`intfloat/multilingual-e5-large`** | 1024 | 다국어 | **[다국어 고성능]** 다국어 벤치마크 상위권 모델. (사용 시 `query:`, `passage:` 접두어 필요) |
    | **`BAAI/bge-m3`** | 1024 | 다국어 | **[한국어 최적]** 한국어 처리 성능이 매우 뛰어나며, 긴 문장(8192 토큰)도 처리 가능. |


`(1) embedding 모델`

- langchain_huggingface 설치 필요

In [9]:
# from langchain_huggingface import HuggingFaceEmbeddings  

# # Hugging Face의 임베딩 모델 생성
# embeddings_gemma = HuggingFaceEmbeddings(
#     model_name="google/embeddinggemma-300m",          # 사용할 모델 이름 - 구글의 경량 임베딩 모델
#     # model_kwargs={'device': 'cuda'}  # GPU 사용시
#     # model_kwargs={'device': 'mps'}   # Mac M1 사용시
# )

# # 임베딩 객체 출력
# embeddings_gemma

In [8]:
# from sentence_transformers import SentenceTransformer

# # 모델 로드
# model = SentenceTransformer("google/embeddinggemma-300m")

# print("모델 로드 완료")

In [10]:
# from transformers import AutoTokenizer, AutoModel
# import torch

# tokenizer = AutoTokenizer.from_pretrained(
#     "google/embeddinggemma-300m"
# )
# model = AutoModel.from_pretrained(
#     "google/embeddinggemma-300m"
# )

# inputs = tokenizer("hello world", return_tensors="pt")
# with torch.no_grad():
#     outputs = model(**inputs)

# embedding = outputs.last_hidden_state.mean(dim=1)

In [ ]:
# intel mac은 langchain_huggingface 설치 불가능
from langchain_community.embeddings import HuggingFaceEmbeddings

# BAAI/bge-m3 임베딩 모델 생성
embeddings_bge = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",           # 다국어 임베딩 모델
    model_kwargs={'device': 'cpu'},     # Intel Mac은 CPU 사용
    encode_kwargs={
        'normalize_embeddings': True,   # 코사인 유사도 최적화
        'batch_size': 32                # 배치 크기
    }
)

# 임베딩 객체 출력
embeddings_bge

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False}) with Transformer model: XLMRobertaModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
), model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True, 'batch_size': 32}, multi_process=False, show_progress=False)

`(2) embed_documents 사용`

In [21]:
# 문서 임베딩
document_embeddings_gemma = embeddings_bge.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_gemma)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_gemma[0])}")
print(document_embeddings_gemma[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.03941444307565689, 0.008764885365962982, -0.01268157921731472, 0.0024531856179237366, -0.008944753557443619, -0.0073836673982441425, -0.005377344321459532, -0.00905588548630476, 0.03291524201631546, 0.006045501213520765, -0.02701292559504509, -0.027740901336073875, 0.0004441232013050467, 0.030136553570628166, 0.017242850735783577, 0.01709035038948059, 0.025524906814098358, -0.021856090053915977, -0.01134131196886301, -0.05702260509133339, -0.0003016882110387087, 0.01354304887354374, -0.007450084201991558, 0.018574390560388565, 0.0028946558013558388, 0.00863062683492899, -0.000744490884244442, -0.02890409529209137, 0.02072780206799507, -0.020500609651207924, 0.008069886825978756, -0.026754183694720268, 0.003963093739002943, -0.016303902491927147, -0.07406218349933624, -0.033650364726781845, -0.023871470242738724, -0.03455007076263428, -0.03478595241904259, 0.00548298005014658, -0.050033554434776306, -0.0028035768773406744, -0.023146899417042732, -0.074

`(3) embed_query 사용`

In [24]:
embedded_query = embeddings_bge.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.03703910484910011, -0.004837999120354652, 0.0029373185243457556, -0.015514618717133999, -0.0009441775036975741, -0.041501615196466446, -0.00657444866374135, 0.011289633810520172, 0.02161404676735401, 0.004928688984364271, -0.020340600982308388, 0.01690521091222763, -0.012874143198132515, 0.005518912337720394, 0.014988325536251068, 0.02422877959907055, 0.007369121070951223, -0.028049904853105545, -0.014938989654183388, -0.051851898431777954, -0.006705061532557011, -0.009251509793102741, -0.016980772837996483, 0.006491508800536394, 0.0529317744076252, 0.0481373555958271, -0.008069530129432678, -0.023171786218881607, 0.01814303919672966, -0.011328132823109627, -0.004240395501255989, -0.0063547054305672646, -0.0022717320825904608, 0.01432944368571043, -0.03563682362437248, -0.008155830204486847, -0.011798221617937088, -0.04542406275868416, -0.04073287174105644, 0.00221388996578753, -0.012132293544709682, 0.017896125093102455, -0.019144678488373756, -0.04192443564534

`(4) 유사도 기반 검색`

In [25]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_gemma, embeddings_model=embeddings_bge) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7269

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7057

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6843



### 3. **Ollama (로컬 실행 최적화)**

LangChain에서 로컬 LLM 및 임베딩 모델을 가장 손쉽게 실행할 수 있는 플랫폼입니다. 외부 API 전송 없이 로컬 자원(GPU/CPU)만 사용하므로 데이터 보안과 비용 절감에 최적화되어 있습니다.

**주요 특징:**
1. **완전한 로컬 실행:** 데이터가 외부로 유출되지 않아 기업 내부(On-premise) 구축에 적합.
2. **빠른 추론 속도:** C++ 기반의 런타임과 양자화(Quantization) 기술로 최적화됨.
3. **간편한 배포:** Docker 기반으로 모델 설치 및 실행이 매우 간단함 (`ollama pull 모델명`).

**사용 시 주의사항:**
1. **서버 실행 필수:** 백그라운드에서 `ollama serve`가 실행 중이어야 함.
2. **리소스 관리:** 고성능 모델(Large) 사용 시 충분한 RAM/VRAM 필요.
3. **API 설정:** LangChain 등에서 호출 시 엔드포인트(`localhost:11434`) 확인 필요.

**대표적인 임베딩 모델 비교:**

| 모델명 (Model Tag) | 차원 | 언어 | 특징 및 추천 용도 |
| :--- | :---: | :---: | :--- |
| **`nomic-embed-text`** | 768 | 영어 | **[Ollama 표준]** 긴 문맥(8192 토큰)을 지원하며, 오픈소스 중 밸런스가 가장 우수함. |
| **`mxbai-embed-large`** | 1024 | 영어 | **[SOTA 성능]** MTEB 리더보드 상위권 모델. 검색 정확도가 매우 높음. |
| **`snowflake-arctic-embed`** | 1024 | 영어 | **[검색 최적화]** Snowflake사가 RAG 및 대규모 검색 작업에 특화하여 설계함. |
| **`bge-m3`** | 1024 | **다국어** | **[한국어 추천]** Ollama에서 사용 가능한 가장 강력한 다국어/한국어 모델. |
| **`all-minilm`** | 384 | 영어 | **[초경량]** 속도가 매우 빠르고 CPU 환경에서도 부담 없이 실행 가능. |

**임베딩 벡터 특성:**
1. **고정 차원:** 모델별로 384~1024의 고정된 벡터 차원을 가짐.
2. **자동 양자화:** 원본 모델을 4비트(Q4_0) 등으로 압축하여 메모리 사용량을 대폭 줄임.

`(1) embedding 모델`

- langchain_ollama 설치 필요

In [27]:
from langchain_ollama import OllamaEmbeddings 

# OllamaEmbeddings 모델 생성
# embeddings_ollama = OllamaEmbeddings(
#     model="nomic-embed-text",          # 사용할 모델 이름
#     base_url="http://localhost:11434"  # Ollama 서버 주소
# )
embeddings_ollama = OllamaEmbeddings(model="bge-m3")

# 임베딩 객체 출력
embeddings_ollama

OllamaEmbeddings(model='bge-m3', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

`(2) embed_documents 사용`

In [29]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_ollama = embeddings_ollama.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_ollama)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_ollama[0])}")
print(document_embeddings_ollama[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.03934538, 0.008694141, -0.012846038, 0.0023583611, -0.00900711, -0.007407391, -0.0055471505, -0.008987892, 0.03278983, 0.006003136, -0.027277984, -0.02791892, 0.0003893945, 0.030142723, 0.017176902, 0.017275523, 0.025456442, -0.021867001, -0.011509322, -0.056959942, -0.0002266645, 0.013493421, -0.007426777, 0.018535968, 0.0029257368, 0.008601946, -0.000817138, -0.029054658, 0.020723483, -0.020637712, 0.008156453, -0.026876222, 0.003632207, -0.016150331, -0.073898315, -0.033678953, -0.023842439, -0.03434298, -0.034607455, 0.0054900963, -0.05010182, -0.0029461132, -0.02311856, -0.075008065, -0.011283377, -0.028918087, -0.034171563, -0.025449311, -0.06137873, 0.013572747, 0.024454584, -0.031444293, 0.050552674, 0.012980497, -0.06397774, 0.027317427, 0.0070649576, 0.010359302, -0.064039595, -0.016206881, -0.020735383, 0.030214, -0.006330392, -0.013136224, -0.0007044801, 0.046923917, 0.007835429, 0.029651977, -0.034804244, -0.03635801, 0.021484395, 0.01769

`(3) embed_query 사용`

In [30]:
embedded_query = embeddings_ollama.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.036819912, -0.004916618, 0.0029082831, -0.015539129, -0.0009142224, -0.04169709, -0.00672218, 0.011316358, 0.021573426, 0.004884537, -0.020496542, 0.016810413, -0.012866592, 0.0054386123, 0.01498064, 0.024293654, 0.0074506067, -0.02797163, -0.015131868, -0.05180663, -0.0065937447, -0.009214115, -0.017141858, 0.006588746, 0.0529069, 0.048049502, -0.00815988, -0.023181217, 0.018123712, -0.011395433, -0.004364532, -0.006346448, -0.0024438666, 0.0143694775, -0.035459716, -0.008202059, -0.011811498, -0.04528384, -0.04064701, 0.002155194, -0.012297963, 0.017717198, -0.0191383, -0.041963276, 0.0009980951, -0.03902422, -0.024457598, -0.02454922, -0.022195984, -0.0044654435, 0.03154465, -0.048738863, 0.018023117, 0.025265386, 0.002350691, 0.04853485, -0.008126998, 0.028096575, -0.073776506, -0.020729942, -0.026363142, -0.0074827964, -0.038786042, -0.017378187, 0.019248726, 0.06371151, 0.020398587, -0.011836012, -0.018532733, -0.040781062, 0.0024798622, 0.041505236, -0.05

`(4) 유사도 기반 검색`

In [31]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_ollama, embeddings_model=embeddings_ollama) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7271

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7051

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6836



# 실험 결과 비교

## 1. 모델 정보

| 모델 구분 | 모델명 | 임베딩 차원 | 실행 환경 |
|:---|:---|:---:|:---|
| OpenAI | text-embedding-3-small | 1536 | API (클라우드) |
| HuggingFace | BAAI/bge-m3 | 1024 | 로컬 (CPU) |
| Ollama | bge-m3 | 1024 | 로컬 서버 |

## 2. 쿼리별 유사도 검색 결과

### 쿼리 1: "인공지능이란 무엇인가요?"

| 모델 | 가장 유사한 문서 | 유사도 |
|:---|:---|:---:|
| OpenAI | 인공지능은 컴퓨터 과학의 한 분야입니다. | 0.7119 |
| HuggingFace | 인공지능은 컴퓨터 과학의 한 분야입니다. | 0.7269 |
| Ollama | 인공지능은 컴퓨터 과학의 한 분야입니다. | 0.7271 |

### 쿼리 2: "딥러닝과 머신러닝의 관계는 어떻게 되나요?"

| 모델 | 가장 유사한 문서 | 유사도 |
|:---|:---|:---:|
| OpenAI | 딥러닝은 머신러닝의 한 종류입니다. | 0.6817 |
| HuggingFace | 딥러닝은 머신러닝의 한 종류입니다. | 0.7057 |
| Ollama | 딥러닝은 머신러닝의 한 종류입니다. | 0.7051 |

### 쿼리 3: "컴퓨터가 이미지를 이해하는 방법은?"

| 모델 | 가장 유사한 문서 | 유사도 |
|:---|:---|:---:|
| OpenAI | 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다. | 0.7052 |
| HuggingFace | 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다. | 0.6843 |
| Ollama | 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다. | 0.6836 |

## 3. 종합 분석

| 지표 | OpenAI | HuggingFace | Ollama |
|:---|:---:|:---:|:---:|
| 평균 유사도 | 0.6996 | 0.7056 | 0.7053 |
| 최고 유사도 | 0.7119 | 0.7269 | 0.7271 |
| 최저 유사도 | 0.6817 | 0.6843 | 0.6836 |

### 주요 발견:
- ✅ 모든 모델이 각 쿼리에 대해 **동일한 문서**를 가장 유사한 것으로 선택
- ✅ HuggingFace와 Ollama(둘 다 bge-m3)가 **거의 동일한 성능** (평균 유사도 0.7056 vs 0.7053)
- ✅ OpenAI는 평균 유사도가 약간 낮지만, 쿼리 3에서는 가장 높은 성능
- ✅ **한국어 처리**에서는 BAAI/bge-m3 모델(HuggingFace, Ollama)이 전반적으로 우수한 성능을 보임

# [실습 프로젝트]
2026.2.21(토) 0h30 금요일 수업 이후 혼자 따라잡아 해보기 ㅜㅜ
(수업 시간 중에는 의존성 설치 등 이슈로 제대로 실습 못 따라함)

In [28]:
# OpenAI 임베딩 모델 import
from langchain_openai import OpenAIEmbeddings

# text-embedding-3-small 모델은 Matryoshka Embedding을 지원하여
# dimensions 파라미터로 임베딩 차원을 조정할 수 있습니다.
# 기본 차원: 1536, 축소 가능 범위: 1~1536

### 1. OpenAI text-embedding-3-small 임베딩 모델을 초기화합니다. 

### 2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델을 생성합니다. 

In [29]:
# OpenAIEmbeddings 모델 생성
embeddings_model_512 = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
)

# 임베딩 객체 출력
embeddings_model_512


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x121d26990>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1211e6050>, model='text-embedding-3-small', dimensions=512, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [33]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model_512.embedding_ctx_length

8191

In [32]:
# 임베딩 모델의 임베딩 차원 확인 - 기본값 (None)
embeddings_model_512.dimensions

512

In [30]:
embeddings_model_1536 = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=1536, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
)

embeddings_model_1536

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x120f017d0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x120e9f090>, model='text-embedding-3-small', dimensions=1536, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [34]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model_1536.embedding_ctx_length

8191

In [31]:
embeddings_model_1536.dimensions

1536

### 3. 아래 주어진 문장들의 임베딩을 생성합니다. (임베딩 차원이 512, 1536인 경우 각각 2개씩 생성)
   - 문장1: "인공지능은 현대 사회를 변화시키고 있다"
   - 문장2: "AI 기술이 우리의 미래를 바꾸고 있다"

In [35]:
# 문서 컬렉션
documents = [
    "인공지능은 현대 사회를 변화시키고 있다",
    "AI 기술이 우리의 미래를 바꾸고 있다"
]

# 문서 임베딩
document_embeddings_512 = embeddings_model_512.embed_documents(documents)
document_embeddings_1536 = embeddings_model_1536.embed_documents(documents)

### 4. 생성된 임베딩의 차원을 출력합니다. (임베딩 차원이 512, 1536인 경우를 각각 출력)

In [37]:
# 512 차원 모델 결과
print("=== 512 차원 모델 ===")
print(f"임베딩 벡터 개수: {len(document_embeddings_512)}")
print(f"임베딩 벡터 차원: {len(document_embeddings_512[0])}")
print(f"첫번째 임베딩 벡터의 첫 10개 값: {document_embeddings_512[0][:10]}")

# 1536 차원 모델 결과
print("\n=== 1536 차원 모델 ===")
print(f"임베딩 벡터 개수: {len(document_embeddings_1536)}")
print(f"임베딩 벡터 차원: {len(document_embeddings_1536[0])}")
print(f"첫번째 임베딩 벡터의 첫 10개 값: {document_embeddings_1536[0][:10]}")

=== 512 차원 모델 ===
임베딩 벡터 개수: 2
임베딩 벡터 차원: 512
첫번째 임베딩 벡터의 첫 10개 값: [0.014563960023224354, 0.019542785361409187, 0.017837805673480034, 0.06378627568483353, 0.030460381880402565, -0.057711392641067505, 0.0002171520027332008, 0.1421293467283249, 0.0032004176173359156, 0.023139001801609993]

=== 1536 차원 모델 ===
임베딩 벡터 개수: 2
임베딩 벡터 차원: 1536
첫번째 임베딩 벡터의 첫 10개 값: [0.010054979473352432, 0.013485969044268131, 0.012345611117780209, 0.04410704970359802, 0.021061910316348076, -0.03996209800243378, 0.00016640545800328255, 0.09852690249681473, 0.002232374157756567, 0.015974923968315125]


### 5. 두 문장 간의 코사인 유사도를 계산합니다. (임베딩 차원이 512, 1536인 경우를 각각 비교)

In [39]:
# 두 문장 간 코사인 유사도 계산
similarity_512 = cosine_similarity(
    [document_embeddings_512[0]],  # 문장1
    [document_embeddings_512[1]]   # 문장2
)[0][0]

similarity_1536 = cosine_similarity(
    [document_embeddings_1536[0]], # 문장1
    [document_embeddings_1536[1]]  # 문장2
)[0][0]

# 결과 비교
print(f"512 차원 모델 - 두 문장 간 유사도: {similarity_512:.4f}")
print(f"1536 차원 모델 - 두 문장 간 유사도: {similarity_1536:.4f}")

512 차원 모델 - 두 문장 간 유사도: 0.5538
1536 차원 모델 - 두 문장 간 유사도: 0.4955


### 6. [추가] 임베딩 차원(512 vs 1536)에 따른 유사도 차이를 분석하고, 어떤 차원이 더 효과적인지 생각해보세요.

In [40]:
print(f"유사도 차이: {abs(similarity_1536 - similarity_512):.4f}")

유사도 차이: 0.0583


#### 실험 결과
- **512 차원**: 유사도 0.5538 (더 높음)
- **1536 차원**: 유사도 0.4955 (더 낮음)
- **차이**: 0.0583

#### 해석

**512 차원 모델의 특성:**
- ✅ 정보 압축이 강함 → 일반적인 의미 파악에 유리
- ✅ 계산 속도가 빠르고 저장 공간 절약
- ⚠️ 세밀한 의미 차이를 구분하는 능력이 상대적으로 낮음

**1536 차원 모델의 특성:**
- ✅ 더 많은 정보를 보존 → 미묘한 의미 차이 포착 가능
- ✅ 복잡한 문맥이나 전문 용어 처리에 유리
- ⚠️ 계산 비용과 저장 공간이 3배

#### 결론

이 실험에서는 512차원이 더 높은 유사도를 보였지만, **유사도가 높다고 해서 반드시 더 효과적인 것은 아닙니다.**

**차원 선택 가이드:**
- **일반적인 RAG 시스템**: 512 차원으로 충분 (비용 효율적)
- **전문 분야/법률/의료**: 1536 차원 권장 (정확도 중요)
- **대량 데이터 처리**: 512 차원 권장 (속도/비용)

**중요:** 실제 프로젝트에서는 다양한 쿼리로 **A/B 테스트**를 수행하여 최적의 차원을 선택해야 합니다.

**추가 고려사항:**
- 512차원에서 높은 유사도: 두 문장의 '일반적인 의미'가 유사함을 의미
- 1536차원에서 낮은 유사도: 표현 방식의 '미묘한 차이'까지 포착한 결과
- 실제로 "인공지능은 현대 사회를 변화시키고 있다"와 "AI 기술이 우리의 미래를 바꾸고 있다"는 유사하지만 동일하지 않은 문장이므로, 어느 쪽이 더 정확한지는 사용 목적에 따라 다릅니다.